# 💰 Precios Mayoristas

Captura completa de la landing **Ventas por Mayor** de Sodimac, con precios B2B.

> 🔄 **Auto-actualizado desde GitHub.** Cada vez que le des Run all, la celda de abajo trae la última versión del código. **No necesitas reinstalar ni actualizar nada manualmente.**

> 💾 **Puedes guardar este notebook en tu Drive sin problema** (File → Save a copy in Drive). El auto-update sigue funcionando porque vive dentro de la celda de bootstrap. Lo único que no debes hacer es editar las celdas \[1\] (bootstrap) o \[2\] (boot).

In [ ]:
# === Auto-update desde GitHub — ¡no toques esta celda! ===
# Siempre trae la última versión de los launchers + engines antes de ejecutar.
import subprocess, sys, os, json, shutil, time
from pathlib import Path

REPO_URL        = "https://github.com/carloscruzerrazuriz/GoWild-Scraper.git"
REPO_BRANCH     = "main"
EXPECTED_SCHEMA = "1.0"  # versión del launcher que este notebook entiende
LOCAL_DIR       = Path("/content/gowild") if Path("/content").exists() else Path.cwd() / "gowild"

def _run(cmd, **kw):
    return subprocess.run(cmd, check=True, capture_output=True, text=True, **kw)

t0 = time.time()
try:
    if LOCAL_DIR.exists() and (LOCAL_DIR / ".git").exists():
        _run(["git", "-C", str(LOCAL_DIR), "fetch", "--quiet", "--depth", "1", "origin", REPO_BRANCH])
        _run(["git", "-C", str(LOCAL_DIR), "reset", "--hard", "--quiet", f"origin/{REPO_BRANCH}"])
        action = "actualizado"
    else:
        if LOCAL_DIR.exists():
            shutil.rmtree(LOCAL_DIR)
        _run(["git", "clone", "--quiet", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(LOCAL_DIR)])
        action = "clonado"

    if str(LOCAL_DIR) not in sys.path:
        sys.path.insert(0, str(LOCAL_DIR))
    # CRÍTICO: limpiar cache de Python para forzar reimport del código recién pulled.
    # Sin esto, un "Run all" sin Restart Kernel usa la versión vieja en memoria.
    for _cached in [_n for _n in list(sys.modules) if _n.startswith(("launchers", "engines"))]:
        del sys.modules[_cached]

    # Instalar/actualizar dependencias desde requirements.txt (silent, idempotente)
    req_file = LOCAL_DIR / "requirements.txt"
    if req_file.exists():
        print(f"📦 Instalando dependencias (primera vez ~30s, después instantáneo)…")
        try:
            _run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req_file)])
            # Playwright necesita además bajar el binario de Chromium (idempotente)
            _run([sys.executable, "-m", "playwright", "install", "chromium"])
        except subprocess.CalledProcessError as e:
            print(f"   ⚠️ pip install reportó error (continuando): {(e.stderr or '').strip()[:200]}")


    # Validación de schema: si cambió, hay que reenviar el .ipynb.
    version_data = json.loads((LOCAL_DIR / "version.json").read_text())
    repo_schema = version_data.get("launcher_schema", "?")
    commit = _run(["git", "-C", str(LOCAL_DIR), "rev-parse", "--short", "HEAD"]).stdout.strip()
    print(f"✅ Repo {action} en {time.time()-t0:.1f}s · version {version_data.get('version','?')} · commit {commit}")

    if repo_schema != EXPECTED_SCHEMA:
        from IPython.display import display, Markdown
        display(Markdown(
            f"### ⚠️ Notebook desactualizado\n\n"
            f"Este notebook usa launcher schema **{EXPECTED_SCHEMA}** pero el repo está en **{repo_schema}**.\n\n"
            f"Solicita la versión nueva del `.ipynb` a quien te lo envió. Mientras tanto, "
            f"el código de abajo puede no funcionar correctamente."
        ))
except subprocess.CalledProcessError as e:
    print("❌ Falló auto-update desde GitHub")
    print("   stderr:", (e.stderr or '').strip())
    raise


In [ ]:
# === Lanzar herramienta (mayoristas) ===
# Esta celda invoca el launcher actualizado desde GitHub.
# Si quieres ver el código fuente: https://github.com/carloscruzerrazuriz/GoWild-Scraper/blob/main/launchers/mayoristas.py
from launchers import boot
boot("mayoristas")
